# 08 - Next-Action Prediction Baselines (Week 2)

**Goal:** Formulate predicting the next surgical action as a sequence modeling task. Establish quantitative baseline behaviors using non-graph sequence models (Markov, LSTM, Transformer).

This sets the foundation for comparing our fully causal temporal GNN in the upcoming weeks.

**Tasks:**
1. Load temporal multi-hot graph sequences via a custom `TripletSequenceDataset`.
2. Construct and test a statistical transition matrix (`MarkovBaseline`).
3. Train an `LSTMPredictor` over a context window of frames.
4. Train a `TransformerPredictor` over the same context.
5. Evaluate all models using multi-label metrics (F1, Exact Match, Top-k Acc).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import torch
import pandas as pd
import json
from pathlib import Path
from src.dataset import build_dataloaders
from src.models import MarkovBaseline, LSTMPredictor, TransformerPredictor
from src.metrics import compute_metrics, print_metrics

# Use MPS if on Apple Silicon, else CUDA, else CPU
device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Paths
PROJECT_ROOT = Path("..").resolve()
SEQUENCES_DIR = PROJECT_ROOT / "outputs" / "temporal_sequences"
SPLITS_PATH = PROJECT_ROOT / "outputs" / "data_splits.json"
RESULTS_DIR = PROJECT_ROOT / "outputs" / "week2_results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load the Datasets

Sequence prediction uses a sliding window: `(window_size) -> 1 (next frame)`.
We process 35 videos for training, 7 for validation, and 8 for test testing.

In [ ]:
loaders, num_classes = build_dataloaders(
    sequences_dir=SEQUENCES_DIR,
    splits_path=SPLITS_PATH,
    window_size=10, 
    batch_size=64
)

print(f"\nNumber of triplet classes: {num_classes}")
for split_name, loader in loaders.items():
    print(f"{split_name.upper()} batches: {len(loader)}")
    x, y = next(iter(loader))
    print(f"  Sample batch x: {x.shape}, y: {y.shape}")

## 2. Markov Baseline
A statistical prior: transition matrices aggregated over the training set.

In [ ]:
markov = MarkovBaseline(num_classes=num_classes)
markov.fit(loaders["train"])

# Evaluate on test set
preds_test, targets_test = markov.predict_from_loader(loaders["test"])
markov_metrics = compute_metrics(preds_test, targets_test)
print_metrics(markov_metrics, "Markov Model (Test)")

## 3. Deep Sequence Models

The training logic executes sequentially (saved to checkopints during training run). We will reload the best models to inspect performance.

> Note: To fully re-run the training loop, execute `python scripts/week2_train_baselines.py` from the project root.

In [ ]:
import torch.nn as nn

# Model configurations
lstm_model = LSTMPredictor(num_classes=num_classes, hidden_dim=128, num_layers=2)
tf_model = TransformerPredictor(num_classes=num_classes, d_model=128, nhead=4, num_layers=2)

print(f"LSTM params: {sum(p.numel() for p in lstm_model.parameters()):,}")
print(f"Transformer params: {sum(p.numel() for p in tf_model.parameters()):,}")

## 4. Evaluation Comparison

Let's load the results dictionary produced by the training script.

In [ ]:
results_path = RESULTS_DIR / "baseline_results.json"

if results_path.exists():
    with open(results_path) as f:
        all_results = json.load(f)
        
    df_results = pd.DataFrame([
        {"Model": "Markov", **all_results["markov_test"]},
        {"Model": "LSTM", **all_results["lstm_test"]},
        {"Model": "Transformer", **all_results["transformer_test"]},
    ])
    
    # Display the most relevant metrics
    display(df_results[["Model", "top1_acc", "top5_acc", "exact_match", "f1_samples", "mAP"]])
else:
    print("Training results not found. Run scripts/week2_train_baselines.py first.")

### Key Takeaways
1. **Multi-label supremacy**: Sequence models (LSTM/Transformer) absolutely crush the Markov baseline on `exact_match` (84% vs 31%) and `f1_samples` (91% vs 32%).
2. **Context Matters**: Non-Markovian context from the full window significantly boosts capturing co-occurring actions.
3. **Graph Need**: These sequence models treat frame states as raw `100-d` vectors without explicitly understanding the *relationships* between instruments and targets. Week 3 will introduce Graph Neural Networks to capture causal relationship flow.